# Generating Mock Data

## imports

In [1]:
# Without the following work-around line, pytorch is incompatible with agama
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


import agama
import numpy as np

import torch
from sbi.inference import NLE
from sbi.utils import BoxUniform
from sbi.inference import likelihood_estimator_based_potential
from sbi.analysis import conditional_pairplot

/Users/andyyu/Documents/Python/SUDS/suds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set units
agama.setUnits(mass=1, length=1, velocity=1)

def make_potential(p_0: float, r_s: float, gamma: float) -> agama.Potential:
    """
    Makes potential according to GNFW profile
    
    - p_0: density normalization
    - r_s: scale radius
    - gamma: inner slope

    Preconditions:
    -   0 <= gamma <= 1
    """

    # Based on GNFW Profile
    param = {
        "type": "Spheroid", 
        "densityNorm": p_0,
        "scaleRadius": r_s,
        "gamma": gamma,
        "beta": 3,
        "alpha": 1
    }

    return agama.Potential(param)

def make_density(r_star: float):
    """
    Creates stellar density distribution according to the 3D Plummer Profile

    - r_star: scale length
    """

    # Based on Plummer profile
    param = {
        "type": "Spheroid",
        "mass": 1,
        "scaleRadius": r_star,
        "gamma": 0,
        "beta": 5,
        "alpha": 2,
    }
    
    return agama.Density(param)


def generate_galaxy(p_0: float, r_s: float, gamma: float, r_star: float, r_a: float):
    """
    Generate the galaxy model given theta

    - p_0: density normalization
    - r_s: scale radius
    - gamma: inner slope
    - r_star: scale length
    - r_a: the radius of transition from isotropic velocity orbits at small radii to radially biased orbits at larger radii (anisotropy radius).

    """

    pot = make_potential(p_0, r_s, gamma)

    df = agama.DistributionFunction(
        type = "QuasiSpherical",
        potential = pot,
        density = make_density(r_star),
        # anisType = "OsipkovMerritt",
        r_a = r_a
    )

    return agama.GalaxyModel(pot, df)

def transform_params(theta: torch.Tensor) -> torch.Tensor:
    """
    transform parameters into correct for generate_galaxy

    - theta: tensor of sampled theta with columns \
        log(p_0), log(r_s), gamma, r_star/r_s, r_a/r_star
    """


    p_0 = 10 ** theta[:,0]
    r_s = 10 ** theta[:,1]
    gamma = theta[:,2]
    r_star = theta[:,3] * r_s
    r_a = theta[:,4] * r_star

    return torch.stack([p_0, r_s, gamma, r_star, r_a], dim=1)


def generate_galaxy_bunch(theta: torch.Tensor) -> torch.Tensor:
    """
    Generate the galaxy model given theta

    - theta: tensor of sampled theta with columns \
        log(p_0), log(r_s), gamma, r_star/r_s, r_a/r_star
    """
    transformed_theta = transform_params(theta)
    
    samples_np = np.vstack([generate_galaxy(*row).sample(1)[0][0] for row in transformed_theta])

    return torch.from_numpy(samples_np).to(torch.float32)  # sbi requires float 32

    
# torch.tensor(generate_galaxy(10e5, 10e2, 1, 0.3, 0.3).sample(1)[0][0])
# theta.shape[0]
# torch.zeros(theta.shape)
# torch.zeros([theta.shape[0], 6])

## Training

In [3]:
agama.setRandomSeed(13)
torch.manual_seed(13)

### Proof of concept

In [4]:
# Check table 1 from Nguyen et al. for prior values

# Create the prior boundary in following order: log(p_0), log(r_s), gamma, r_star/r_s, r_a/r_star

low = torch.tensor([5, -1, -1, 0.2, 0.5])
high = torch.tensor([8, 0.7, 2, 1, 2])

# Create uniform distribution
prior = BoxUniform(low=low, high=high)

# Sample from said distribution
n_samples = 100
theta = prior.sample((n_samples,))

# Generate the data
x = generate_galaxy_bunch(theta)

In [5]:
# Set up and train model
inference = NLE(prior=prior)
likelihood_estimator = inference.append_simulations(theta, x).train()

/Users/andyyu/Documents/Python/SUDS/suds/lib/python3.11/site-packages/sbi/inference/trainers/base.py:296: UserWarning: Data has extreme outliers in dimension(s) [0, 1, 2, 3, 4, 5] (beyond 10.0x IQR from quartiles). This may cause precision loss during z-scoring, where distinct values become indistinguishable. Consider removing outliers from your data or setting `z_score_x='none'` (though this may affect training).
  warn_if_invalid_for_zscoring(x)


 Neural network successfully converged after 220 epochs.

In [6]:
test_theta = torch.tensor([[5, 0, 1, 0.5, 0.5],])
test_theta

tensor([[5.0000, 0.0000, 1.0000, 0.5000, 0.5000]])

In [10]:
transformed_theta = transform_params(test_theta)
    
transformed_theta[0]

tensor([1.0000e+05, 1.0000e+00, 1.0000e+00, 5.0000e-01, 2.5000e-01])

In [ ]:
generate_galaxy(*transformed_theta[0])

In [ ]:
[generate_galaxy(*row).sample(1)[0][0] for row in transformed_theta]

In [ ]:
samples_np = np.vstack([generate_galaxy(*row).sample(1)[0][0] for row in transformed_theta])

# return torch.from_numpy(samples_np).to(torch.float32)  # sbi requires float 32

In [ ]:
test_x = generate_galaxy_bunch(test_theta)


In [ ]:
test_theta = torch.tensor([[5, 0, 1, 0.5, 0.5],])
test_x = generate_galaxy_bunch(test_theta)

likelihood_estimator.log_prob(test_x, condition=test_theta)



In [ ]:
k = torch.tensor([theta[0].tolist()])
generate_galaxy_bunch(k)

tensor([[ 5.2755, -0.1851,  1.4317,  0.2121,  0.5230]])

In [2]:
torch.tensor([5,0,1,0.5,0.5])

tensor([5.0000, 0.0000, 1.0000, 0.5000, 0.5000])

In [4]:
generate_galaxy_bunch(torch.tensor([[5,0,1,0.5,0.5],]))

tensor([[-1.6040,  0.1722, -0.1127,  1.1448,  0.0425,  0.1383]])

In [7]:
# likelihood_estimator
# (torch.tensor([-4.1985e-01, -1.2428e-02,  8.9781e-02,  9.5296e+01, -5.0721e+01, -4.3831e+01]),
#                               context=(10e5, 10e2, 1, 0.3, 0.3))
# likelihood_estimator_based_potential(likelihood_estimator, prior,torch.tensor([-4.1985e-01, -1.2428e-02,  8.9781e-02,  9.5296e+01, -5.0721e+01, -4.3831e+01]))

In [8]:
# posterior = inference.build_posterior(likelihood_estimator)
# posterior.sample((10,), x=torch.tensor([-4.1985e-01, -1.2428e-02,  8.9781e-02,  9.5296e+01, -5.0721e+01,
#         -4.3831e+01]))

In [9]:
# conditional_pairplot(likelihood_estimator,
#                      torch.tensor([6, 0, 1, 0.5, 0.5]),
#                      )